# Pass 1 — Extract Objectives from Every Column

**Goal:** For each fund, send ALL non-empty objective columns in a single API call.  
The LLM extracts objectives **per column independently** — no cross-column reasoning yet.  

**Output:** One row per fund, with per-column extraction results stored as JSON.

In [59]:
import pandas as pd
from tqdm import tqdm
import time, os, json, anthropic
from pathlib import Path

with open("Claude_API.txt", "r") as file:
    api_key = file.read().strip()
os.environ['ANTHROPIC_API_KEY'] = api_key

config = {}
with open("File_Directory.txt", "r") as file:
    for line in file:
        if ":" in line:
            key, value = line.split(":", 1)
            config[key.strip()] = value.strip()

INPUT_FILE = Path(config["Input"])
OUTPUT_DIR = Path(config["Output"])

MODEL = "claude-sonnet-4-6"  # UPDATE as needed

OBJECTIVE_COLUMNS = [
    'PRIIPS KID Objective',
    'KIID Objective/Investment Policy',
    'Prospectus Objective',
    'Investment Strategy - English',
    'PRIIPS KID Objective - Danish',
    'PRIIPS KID Objective - Dutch',
    'PRIIPS KID Objective - Finnish',
    'PRIIPS KID Objective - French',
    'PRIIPS KID Objective - German',
    'PRIIPS KID Objective - Italian',
    'PRIIPS KID Objective - Norwegian',
    'PRIIPS KID Objective - Portuguese',
    'PRIIPS KID Objective - Spanish',
    'PRIIPS KID Objective - Swedish',
    'KIID Objective/Investment Policy - German',
    'KIID Objective/Investment Policy - French',
    'KIID Objective/Investment Policy - Italian',
    'KIID Objective/Investment Policy - Spanish',
    'KIID Objective/Investment Policy - Norwegian',
    'KIID Objective/Investment Policy - Swedish',
    'KIID Objective/Investment Policy - Finnish',
    'KIID Objective/Investment Policy - Portuguese',
    'KIID Objective/Investment Policy - Danish',
    'Investment Strategy - Danish',
    'Investment Strategy - Finnish',
    'Investment Strategy - French',
    'Investment Strategy - German',
    'Investment Strategy - Italian',
    'Investment Strategy - Norwegian',
    'Investment Strategy - Portuguese',
    'Investment Strategy - Spanish',
    'Investment Strategy - Swedish',
    'Strategy Description'
]

In [70]:
PASS1_SYSTEM_PROMPT = """You are extracting fund objectives from regulatory disclosure text for European mutual funds.


CORE EXTRACTION PRINCIPLE:

Extract WHAT the fund aims to achieve for investors.
Strip everything that describes HOW.

HOW covers:
- Implementation: what the fund invests in, asset allocation,
  geographic or sector focus, types of securities
- Mechanism: how the goal is pursued ("by investing in...",
  "through active management...", "through exposure to...",
  "by selecting...", "through a strategy of...",
  "on a [diversified] portfolio of [asset class]...")
- Measurement: how performance, ESG scores, or carbon intensity
  are calculated or tracked
- Regulation: SFDR compliance language, minimum allocation
  requirements, mandatory portfolio composition rules
- Constraints: self-imposed rules the portfolio must satisfy —
  ESG score thresholds, carbon intensity limits, minimum shares
  of sustainable investments
- Exclusions: types of firms, activities, or industries the fund
  avoids

The HOW list above is illustrative, not exhaustive. The underlying
principle: a clause is HOW if removing it would not change what
the fund is trying to achieve for investors, only the manner in
which or where it pursues it.

You will receive MULTIPLE columns of text for a single fund. Extract objectives from EACH column INDEPENDENTLY.
Do NOT cross-reference between columns. Treat each column as a standalone source.

WHAT IS A FUND OBJECTIVE:
The fund objective is the statement of what the fund aims to achieve for its investors — its goal or intended outcome.
Examples: long-term capital growth, regular income, maximizing total returns, beating a benchmark, matching an index.

MULTIPLE OBJECTIVES:
There may be more than one objective per column. Extract ALL and label them separately.
Split objectives when joined by "and", "and/or", "while", "which also", "that also", or similar.
For "and/or" constructions, treat each alternative as a distinct objective.
If one part states a financial goal and another states a sustainability goal, split them.
Examples:
- "provide income and moderate capital growth" → two objectives: "provide income" and "achieve moderate capital growth"
- "exceed the performance of the index while maintaining a higher ESG score" → two objectives
- "achieve capital growth and/or continuous returns" → two objectives: "achieve capital growth" and "achieve continuous returns"
- "long-term capital growth and the generation of a market-appropriate return" → two objectives: "long-term capital growth" and "achieve a market-appropriate return"
- "targeting a significantly reduced risk of loss and significantly lower volatility compared to the general equity market" → two objectives: "reduce risk of capital loss" and "maintain lower volatility than the general equity market"

HOW clauses do not appear as coordinate "and" clauses parallel to the main objective. When a phrase follows "and" as the second part of a compound objective statement, treat it as a second WHAT candidate — evaluate it independently before applying any stripping.

Nominalized goal phrases — "the generation of X", "the achievement of X", "the maximisation of X", "the preservation of X", "the realisation of X" — that follow "and" are noun-form objective statements. Rephrase them as clean verb-form objectives: "generation of a market-appropriate return" → "achieve a market-appropriate return".

"while" clauses — critical distinction:
- "while REDUCING / MINIMISING risk" = a second objective → split and extract separately
- "while ACCEPTING / TOLERATING [higher] risk" = a risk tolerance descriptor → exclude entirely

"Income and capital growth" is always two distinct objectives — income refers to cash distributions or yield; capital growth refers to price appreciation. Always split.
- "achieve income and capital growth over the medium to long term" → two objectives: "achieve income" and "achieve capital growth over the medium to long term"
Exception: When income and capital growth appear inside a HOW phrase ("through a combination of capital growth and income"), strip the entire HOW phrase — do not split.
- "achieve a total return through a combination of capital growth and income" → one objective: "achieve a total return"

For what qualifies as "sustainable" vs "sustainable_disclosure", see SUSTAINABILITY CONTENT below.

DO NOT INCLUDE:
- Investment policy/strategy: what the fund invests in, how securities are selected, asset allocation
- Mechanism or investment vehicle: "by investing in...", "through active management...", "on a portfolio of..."
- Company-activity descriptions: "invest in companies that...", "companies whose products..." — even if labeled "sustainable investment objective", extract only what the fund itself aims to achieve
- Investment philosophy statements or general descriptions of the fund's investment approach
- "while taking into account ESG criteria" or "taking into account the risk level" = not an objective
- Risk information, distribution/dividend policy
- Benchmark references used solely for comparison (not as a target to beat)
- Duplicate objectives within the same column

SUSTAINABILITY CONTENT — WHAT TO EXTRACT VS WHAT TO EXCLUDE:

This is the most important judgment call in the extraction. Apply these rules in order:

The key test: Does the text describe something the fund COMMITS TO ACHIEVING (an outcome, a target, a minimum allocation) or something the fund TAKES INTO ACCOUNT (a process, a consideration, a methodology)? Extract the former, exclude the latter — but also apply the sustainable vs sustainable_disclosure distinction in Step 2 below.

Step 1 — Is it pure SFDR Article 8/9 boilerplate?
These phrases are required regulatory language and are NEVER an objective on their own:
- "promotes environmental and/or social characteristics"
- "is promoting ESG characteristics"
- "is classified as Article 8 under SFDR"
If the text contains ONLY this boilerplate with no additional specifics, there is no sustainable objective.

Step 2 — Does the text go beyond boilerplate with a specific sustainable commitment?
If yes, determine whether it is a genuine independently-set objective or a regulatory disclosure statement — and classify accordingly as "sustainable" or "sustainable_disclosure".

Use "sustainable" when the fund has clearly set a specific target of its own choosing:
- A named ESG score target relative to a benchmark or universe
- A specific carbon reduction target the fund has set as a goal
- Named environmental/social outcomes the fund commits to achieving (GHG reductions, biodiversity, SDG contribution)
- Specific solidarity commitments with a named % allocation
- Sector exclusions framed as a fund-level goal (tobacco, fossil fuels, weapons)

EXTRACT as "sustainable":
- "contribute to reducing greenhouse gas emissions" → sustainable
- "higher ESG score than the index" → sustainable
- "lower carbon intensity than the benchmark index" → sustainable
- "positive impact on environment and social objectives" → sustainable
- "solidarity investments of 5-10% in approved solidarity enterprises" → sustainable
- "integrating criteria for good governance and sustainable development" → sustainable

Use "sustainable_disclosure" when the text uses regulatory language that may be a mandatory disclosure rather than a fund-chosen objective — specifically when it:
- References any regulation, law, or article by name or number ("as defined under SFDR", "in accordance with Article 8/9", "pursuant to Regulation (EU) 2019/2088", "pursuant to Article L.3332-17-1 of the Labour Code", "under Article X of [any law]")
- States only a generic minimum allocation without a specific fund-set % figure (e.g. "a minimum share" with no number)
- Reads as a portfolio composition rule rather than a stated goal

EXTRACT as "sustainable_disclosure":
- "invests at least X% of assets in Sustainable Investments, as defined under SFDR" → sustainable_disclosure (regulatory reference language)
- "maintains a minimum share of sustainable investments" [no specific % stated] → sustainable_disclosure (generic minimum, no fund-set target)
- "Between 5% and 10% of assets are invested in approved solidarity enterprises pursuant to Article L.3332-17-1 of the Labour Code" → sustainable_disclosure (legal article citation)

Do not extract process descriptions: "taking into account ESG criteria", "considering sustainability risks", "ESG integration", "employs ESG criteria in stock selection" — these describe methodology, not outcomes. Generic "promotes environmental and/or social characteristics" is covered by Step 1 above.

Step 3 — Does the fund explicitly disclaim sustainable objectives?
If the text states "the fund does not have sustainable investment as its objective" and the sustainability content is framed purely as an approach or consideration, do not extract a sustainable objective.

Worked examples:
- "The fund promotes environmental and social characteristics and maintains a minimum share of sustainable investments in accordance with Article 8 of the EU SFDR." → EXTRACT as SUSTAINABLE_DISCLOSURE; objective_text: "maintain a minimum share of sustainable investments"
- "The objective is to achieve outperformance while integrating criteria for good governance and sustainable development." → TWO objectives: (1) financial: "outperform the benchmark", (2) sustainable: "integrate criteria for good governance and sustainable development"
- "The fund invests 5-10% of its assets in approved solidarity enterprises." → EXTRACT as SUSTAINABLE; objective_text: "invest 5-10% of assets in solidarity enterprises"
- "The Sub-Fund invests at least 50% of assets in Sustainable Investments, as defined under SFDR." → EXTRACT as SUSTAINABLE_DISCLOSURE
- "The fund targets a carbon footprint at least 5% lower than the Index." → EXTRACT as SUSTAINABLE; objective_text: "maintain a carbon footprint at least 5% lower than the Index"
- "The fund invests in companies whose products contribute to the SDGs." → Do NOT extract — company description, not fund objective.

TIME HORIZON:
Include SPECIFIC time horizons that represent a fund's stated investment horizon: "long-term", "medium-term", "over 5 years", "over a rolling 3-year period".
Exclude GENERIC regulatory language that conveys no fund-specific information: "over the recommended investment period", "over the recommended holding period", "over a multi-year period". These are standard disclosure phrasing, not fund-chosen commitments.

EXTRACTION RULES:
1. Write a concise, clean English statement of the objective — paraphrase if needed to remove scaffolding language and HOW content. Do not copy the full source sentence.
2. Scaffolding to strip from objective_text: "The fund aims to", "The investment objective is to", "The objective of the fund is to", "The Sub-Fund seeks to" — start directly with the goal verb or noun.
3. Record the verbatim source text in source_text — this is 1-3 sentences copied exactly from the source column, in the original language, showing where the objective was identified.
4. Classify each objective as "financial", "sustainable", or "sustainable_disclosure".
5. If no objective can be identified in a column, return an empty list for that column.
6. Detect the language of each column and record it.

OUTPUT FORMAT:
Return a JSON object where each key is the exact column name, and the value is:
{
  "language": "English" or "French" or "German" etc.,
  "objectives": [
    {
      "objective_text": "concise English statement of the goal — no full sentences, no scaffolding, no HOW",
      "source_text": "verbatim 1-3 sentence excerpt from source in original language",
      "objective_type": "financial" or "sustainable" or "sustainable_disclosure"
    }
  ]
}

If a column has no identifiable objective:
{
  "language": "English",
  "objectives": []
}

IMPORTANT: When extracting verbatim source_text that contains quotation marks (including German \u201e...\u201c quotes, French \u00ab...\u00bb quotes, or any other quotation marks), replace them with single quotes. This is critical to ensure valid JSON output.

"""

In [71]:
PASS1_FEW_SHOT = [
    {
        "fund_name": "Example Multi-Column Fund",
        "columns": {
            "PRIIPS KID Objective": "The Fund aims to maximise the return on your investment through a combination of capital growth and income on the Fund's assets and invest in a manner consistent with the principles of environmental, social and governance (ESG) investing. The Fund invests globally at least 70% of its total assets in the equity securities of companies the main business of which is financial services.",
            "PRIIPS KID Objective - French": "Le Fonds vise \u00e0 maximiser le rendement de votre investissement par une combinaison de croissance du capital et de revenus sur les actifs du Fonds et \u00e0 investir d'une mani\u00e8re conforme aux principes de l'investissement environnemental, social et de gouvernance (ESG). Le Fonds investit \u00e0 l'\u00e9chelle mondiale au moins 70 % de son actif total dans les titres de participation de soci\u00e9t\u00e9s dont l'activit\u00e9 principale est les services financiers."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "maximise the return on your investment",
                        "source_text": "The Fund aims to maximise the return on your investment through a combination of capital growth and income on the Fund's assets and invest in a manner consistent with the principles of environmental, social and governance (ESG) investing.",
                        "objective_type": "financial"
                    }
                ]
            },
            "PRIIPS KID Objective - French": {
                "language": "French",
                "objectives": [
                    {
                        "objective_text": "maximise the return on your investment",
                        "source_text": "Le Fonds vise \u00e0 maximiser le rendement de votre investissement par une combinaison de croissance du capital et de revenus sur les actifs du Fonds.",
                        "objective_type": "financial"
                    }
                ]
            }
        }
    },
    {
        "fund_name": "Example Sustainability Split Fund",
        "columns": {
            "PRIIPS KID Objective": "The fund seeks to achieve capital growth and to outperform the benchmark. The fund's sustainable investment objective is to contribute to reducing greenhouse gas emissions. The fund also aims to have long-term positive impact on environment and social objectives."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "achieve capital growth",
                        "source_text": "The fund seeks to achieve capital growth and to outperform the benchmark.",
                        "objective_type": "financial"
                    },
                    {
                        "objective_text": "outperform the benchmark",
                        "source_text": "The fund seeks to achieve capital growth and to outperform the benchmark.",
                        "objective_type": "financial"
                    },
                    {
                        "objective_text": "contribute to reducing greenhouse gas emissions",
                        "source_text": "The fund's sustainable investment objective is to contribute to reducing greenhouse gas emissions.",
                        "objective_type": "sustainable"
                    },
                    {
                        "objective_text": "have long-term positive impact on environment and social objectives",
                        "source_text": "The fund also aims to have long-term positive impact on environment and social objectives.",
                        "objective_type": "sustainable"
                    }
                ]
            }
        }
    },
    {
        "fund_name": "Example Norwegian Fund",
        "columns": {
            "PRIIPS KID Objective": "M\u00e5lsetting\n\nFondets m\u00e5lsetting er \u00e5 skape h\u00f8yest mulig relativ avkastning mot referanseindeksen, MSCI World AC, Net Total Return (m\u00e5lt i NOK).\n\nFondet skal investere i selskaper globalt som har l\u00f8sninger p\u00e5 FN's b\u00e6rekraftsm\u00e5l og dermed bidrar til omstillingen til et mer b\u00e6rekraftig samfunn."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "Norwegian",
                "objectives": [
                    {
                        "objective_text": "create the highest possible relative return against the benchmark index, MSCI World AC, Net Total Return (measured in NOK)",
                        "source_text": "Fondets m\u00e5lsetting er \u00e5 skape h\u00f8yest mulig relativ avkastning mot referanseindeksen, MSCI World AC, Net Total Return (m\u00e5lt i NOK).",
                        "objective_type": "financial"
                    }
                ]
            }
        }
    },
    {
        "fund_name": "Example No-Objective Fund",
        "columns": {
            "PRIIPS KID Objective": "Management objective: Management takes as reference the profitability of the EUROSTOXX 50 Index, solely for informational or comparative purposes. Investment policy: Will invest more than 75% of total exposure in equity assets of European issuers."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": []
            }
        }
    },
    {
        "fund_name": "Example Company-Activity Exclusion Fund",
        "columns": {
            "KIID Objective/Investment Policy": "The fund aims to provide capital growth over the long term (5 years or more) by investing in US companies whose products and services are considered by the investment manager as contributing to positive environmental or social change and thereby have an impact on the development of a sustainable global economy."
        },
        "response": {
            "KIID Objective/Investment Policy": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "provide capital growth over the long term (5 years or more)",
                        "source_text": "The fund aims to provide capital growth over the long term (5 years or more) by investing in US companies whose products and services are considered by the investment manager as contributing to positive environmental or social change.",
                        "objective_type": "financial"
                    }
                ]
            }
        }
    },
    {
        "fund_name": "Example And-Or Split Fund",
        "columns": {
            "PRIIPS KID Objective": "The Fund aims to achieve capital growth and/or continuous returns by investing in a diversified portfolio of global equities."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "achieve capital growth",
                        "source_text": "The Fund aims to achieve capital growth and/or continuous returns by investing in a diversified portfolio of global equities.",
                        "objective_type": "financial"
                    },
                    {
                        "objective_text": "achieve continuous returns",
                        "source_text": "The Fund aims to achieve capital growth and/or continuous returns by investing in a diversified portfolio of global equities.",
                        "objective_type": "financial"
                    }
                ]
            }
        }
    },
    {
        "fund_name": "Example Nominalized Second Objective",
        "columns": {
            "PRIIPS KID Objective": "The investment objective of the fund is long-term capital growth and the generation of a market-appropriate return."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "long-term capital growth",
                        "source_text": "The investment objective of the fund is long-term capital growth and the generation of a market-appropriate return.",
                        "objective_type": "financial"
                    },
                    {
                        "objective_text": "achieve a market-appropriate return",
                        "source_text": "The investment objective of the fund is long-term capital growth and the generation of a market-appropriate return.",
                        "objective_type": "financial"
                    }
                ]
            }
        }
    },
    {
        "fund_name": "Example Income And Capital Growth Split",
        "columns": {
            "PRIIPS KID Objective": "The Fund aims to achieve income and capital growth over the medium to long term."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "achieve income over the medium to long term",
                        "source_text": "The Fund aims to achieve income and capital growth over the medium to long term.",
                        "objective_type": "financial"
                    },
                    {
                        "objective_text": "achieve capital growth over the medium to long term",
                        "source_text": "The Fund aims to achieve income and capital growth over the medium to long term.",
                        "objective_type": "financial"
                    }
                ]
            }
        }
    },
    {
        "fund_name": "Example Sustainable Disclosure Fund",
        "columns": {
            "PRIIPS KID Objective": "The fund seeks to achieve long-term capital growth. The Sub-Fund invests at least 50% of assets in Sustainable Investments, as defined under SFDR.",
            "PRIIPS KID Objective - French": "Le fonds cherche \u00e0 r\u00e9aliser une croissance du capital \u00e0 long terme. Le Sous-Fonds investit au moins 50% de ses actifs dans des Investissements durables, tels que d\u00e9finis par le SFDR."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "achieve long-term capital growth",
                        "source_text": "The fund seeks to achieve long-term capital growth.",
                        "objective_type": "financial"
                    },
                    {
                        "objective_text": "invest at least 50% of assets in Sustainable Investments, as defined under SFDR",
                        "source_text": "The Sub-Fund invests at least 50% of assets in Sustainable Investments, as defined under SFDR.",
                        "objective_type": "sustainable_disclosure"
                    }
                ]
            },
            "PRIIPS KID Objective - French": {
                "language": "French",
                "objectives": [
                    {
                        "objective_text": "achieve long-term capital growth",
                        "source_text": "Le fonds cherche \u00e0 r\u00e9aliser une croissance du capital \u00e0 long terme.",
                        "objective_type": "financial"
                    },
                    {
                        "objective_text": "invest at least 50% of assets in Sustainable Investments, as defined under SFDR",
                        "source_text": "Le Sous-Fonds investit au moins 50% de ses actifs dans des Investissements durables, tels que d\u00e9finis par le SFDR.",
                        "objective_type": "sustainable_disclosure"
                    }
                ]
            }
        }
    }
]

In [62]:
import re

def robust_json_parse(text):
    if not isinstance(text, str):
        return None
    cleaned = text.strip()
    
    # 1. Strip markdown fences
    if cleaned.startswith("```"):
        cleaned = re.sub(r'^```\w*\n?', '', cleaned)
        cleaned = re.sub(r'\n?```\s*$', '', cleaned)
        cleaned = cleaned.strip()
    
    # 2. Replace smart quotes with unicode escapes (always, not just after fences)
    cleaned = cleaned.replace('„', '\\u201E')
    cleaned = cleaned.replace('\u201c', '\\u201C')
    cleaned = cleaned.replace('\u201d', '\\u201D')
    cleaned = cleaned.replace('«', '\\u00AB')
    cleaned = cleaned.replace('»', '\\u00BB')
    cleaned = cleaned.replace('‚', '\\u201A')
    cleaned = cleaned.replace('\u2018', '\\u2018')
    cleaned = cleaned.replace('\u2019', '\\u2019')
    
    # 3. Try direct parse
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # 4. Fix unescaped control characters
    def fix_strings(match):
        s = match.group(0)
        s = s.replace('\n', '\\n')
        s = s.replace('\r', '\\r')
        s = s.replace('\t', '\\t')
        return s
    fixed = re.sub(r'"(?:[^"\\]|\\.)*"', fix_strings, cleaned, flags=re.DOTALL)
    try:
        return json.loads(fixed)
    except json.JSONDecodeError:
        pass

    # 5. Last resort — extract outermost { }
    brace_match = re.search(r'\{.*\}', fixed, re.DOTALL)
    if brace_match:
        try:
            return json.loads(brace_match.group(0))
        except json.JSONDecodeError:
            pass

    return None

In [63]:
def get_nonempty_columns(row, objective_columns):
    """Return dict of only columns that have real content (skip empty/NA)."""
    columns = {}
    for col in objective_columns:
        if col in row.index:
            value = row[col]
            if pd.notna(value) and str(value).strip() not in ['-', 'Not available', '']:
                columns[col] = str(value)
    return columns


def pass1_extract(fund_name, fund_id, columns_dict):
    """Send all non-empty columns for one fund; get per-column extractions back."""
    if not columns_dict:
        return {"_error": "No non-empty columns available"}

    columns_text = "\n\n".join(
        [f"=== Column: {col} ===\n{val}" for col, val in columns_dict.items()]
    )

    user_prompt = f"""Fund ID: {fund_id}
Fund Name: {fund_name}

{columns_text}"""

    messages = []
    for ex in PASS1_FEW_SHOT:
        ex_text = "\n\n".join(
            [f"=== Column: {col} ===\n{val}" for col, val in ex["columns"].items()]
        )
        messages.append({"role": "user", "content": f"Fund Name: {ex['fund_name']}\n\n{ex_text}"})
        messages.append({"role": "assistant", "content": json.dumps(ex["response"], indent=2)})

    messages.append({"role": "user", "content": user_prompt})

    try:
        client = anthropic.Anthropic()
        response = client.messages.create(
            model=MODEL,
            max_tokens=8000,
            temperature=0,
            system=PASS1_SYSTEM_PROMPT,
            messages=messages
        )
        print(f"   [{fund_name}] tokens — in: {response.usage.input_tokens}, out: {response.usage.output_tokens}")

        text = response.content[0].text
        parsed = robust_json_parse(text)
        if parsed is not None:
            return parsed
        return {"_error": f"JSON parse error after all attempts: {text[:300]}"}

    except Exception as e:
        print(f"   Error for {fund_name}: {e}")
        return {"_error": str(e)}

In [64]:
# === LOAD DATA ===
print("Loading data...")
df = pd.read_excel(INPUT_FILE)
print(f"  Total funds: {len(df)}, Columns: {len(df.columns)}")

Loading data...
  Total funds: 5680, Columns: 133


In [65]:
# Adjust sample size as needed: df.sample(n=10, random_state=0) for quick test
df_sample = df.sample(n=100, random_state=30)
#df_sample = df  # Use full dataset

In [72]:
target_ids = [
    "FSGBR0532X", "FSGBR0672S", "FSGBR0585W", "FS00009RRM",
    "FS0000CPCP", "FSGBR058XQ", "FS00009X1T", "FSUSA091TW",
    "FS0000I07Y", "FSUSA08S8L", "FSGBR0580H", "FSGBR054X3",
    "FSGBR06L48", "FSGBR05518", "FSUSA08AT2", "FS0000G8MC",
    "FS0000IINF", "FS0000BTHE", "FS0000FPTR", "FS0000HHY1",
    "FS0000GSO0", "FS0000J9FE"
]

df_sample = df[df['FundId'].isin(target_ids)].copy()

In [73]:
df_sample.to_excel("df_sample.xlsx", index=False)

In [74]:
# === RUN PASS 1 ===
pass1_results = []

for idx in tqdm(range(len(df_sample)), desc="Pass 1 — Extract"):
    row = df_sample.iloc[idx]
    fund_id = row['FundId']
    fund_name = row['Name']

    columns_dict = get_nonempty_columns(row, OBJECTIVE_COLUMNS)
    result = pass1_extract(fund_name, fund_id, columns_dict)

    pass1_results.append({
        'FundId': fund_id,
        'Fund_Name': fund_name,
        'columns_sent': list(columns_dict.keys()),
        'num_columns_sent': len(columns_dict),
        'pass1_raw': result  # full per-column JSON
    })

    if idx > 0 and idx % 50 == 0:
        time.sleep(0.5)

pass1_df = pd.DataFrame(pass1_results)
print(f"\nPass 1 complete: {len(pass1_df)} funds processed")

Pass 1 — Extract:   5%|▍         | 1/22 [00:08<03:02,  8.69s/it]

   [BlackRock Sysmc Eq Fac Pl D EUR H Acc] tokens — in: 7402, out: 687


Pass 1 — Extract:   9%|▉         | 2/22 [00:13<02:06,  6.32s/it]

   [Cardif BNPP IP Smid Cap Euro] tokens — in: 5446, out: 345


Pass 1 — Extract:  14%|█▎        | 3/22 [00:19<01:54,  6.02s/it]

...

   [Partners Group Direct Eq II Eltif I(USD)] tokens — in: 12822, out: 1131


Pass 1 — Extract:  95%|█████████▌| 21/22 [04:08<00:12, 12.15s/it]

   [Redwheel Global Intrinsic Val I GBP Acc] tokens — in: 5986, out: 296


Pass 1 — Extract: 100%|██████████| 22/22 [04:23<00:00, 11.98s/it]

   [UBS (Lux) Eq Fd EM Sst Ldrs (USD) P] tokens — in: 14245, out: 1324

Pass 1 complete: 22 funds processed


In [75]:
# === FLATTEN FOR INSPECTION ===
# Create a human-readable summary alongside the raw JSON

summary_rows = []
for _, row in pass1_df.iterrows():
    raw = row['pass1_raw']
    if '_error' in raw:
        summary_rows.append({
            'FundId': row['FundId'],
            'Fund_Name': row['Fund_Name'],
            'total_objectives_found': 0,
            'columns_with_objectives': 0,
            'error': raw['_error']
        })
        continue

    total_obj = 0
    cols_with_obj = 0
    for col_name, col_data in raw.items():
        if isinstance(col_data, dict) and 'objectives' in col_data:
            n = len(col_data['objectives'])
            total_obj += n
            if n > 0:
                cols_with_obj += 1

    summary_rows.append({
        'FundId': row['FundId'],
        'Fund_Name': row['Fund_Name'],
        'num_columns_sent': row['num_columns_sent'],
        'total_objectives_found': total_obj,
        'columns_with_objectives': cols_with_obj,
        'error': None
    })

summary_df = pd.DataFrame(summary_rows)
print("PASS 1 SUMMARY:")
print(f"  Funds processed: {len(summary_df)}")
print(f"  Funds with errors: {summary_df['error'].notna().sum()}")
print(f"  Funds with ≥1 objective: {(summary_df['total_objectives_found'] > 0).sum()}")
print(f"  Avg objectives per fund: {summary_df['total_objectives_found'].mean():.1f}")
print(f"  Avg columns with objectives: {summary_df['columns_with_objectives'].mean():.1f}")

PASS 1 SUMMARY:
  Funds processed: 22
  Funds with errors: 0
  Funds with ≥1 objective: 22
  Avg objectives per fund: 8.2
  Avg columns with objectives: 5.4


In [76]:
# === SAVE PASS 1 OUTPUT ===
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M")

# Save the raw results (pass1_raw as JSON string for portability)
output_df = pass1_df.copy()
output_df['pass1_raw'] = output_df['pass1_raw'].apply(json.dumps)
output_df['columns_sent'] = output_df['columns_sent'].apply(json.dumps)

p1_filename = f'Pass1_Extract_{len(pass1_df)}_funds_{timestamp}.xlsx'
p1_path = os.path.join(OUTPUT_DIR, p1_filename)
output_df.to_excel(p1_path, index=False, engine='openpyxl')
print(f"Saved: {p1_filename}")
print(f"  → Use this file as input to Pass 2")

Saved: Pass1_Extract_22_funds_20260608_1456.xlsx
  → Use this file as input to Pass 2


: 

: 